In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import joblib
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import KFold, cross_validate

In [2]:
RUN_OLD_MODELS = False

### Data Investigation

In [3]:
df = pd.read_csv("data/NYC.csv")

In [4]:
df.shape

(1458644, 11)

In [5]:
df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 1458644 entries, 0 to 1458643
Data columns (total 11 columns):
 #   Column              Non-Null Count    Dtype  
---  ------              --------------    -----  
 0   id                  1458644 non-null  str    
 1   vendor_id           1458644 non-null  int64  
 2   pickup_datetime     1458644 non-null  str    
 3   dropoff_datetime    1458644 non-null  str    
 4   passenger_count     1458644 non-null  int64  
 5   pickup_longitude    1458644 non-null  float64
 6   pickup_latitude     1458644 non-null  float64
 7   dropoff_longitude   1458644 non-null  float64
 8   dropoff_latitude    1458644 non-null  float64
 9   store_and_fwd_flag  1458644 non-null  str    
 10  trip_duration       1458644 non-null  int64  
dtypes: float64(4), int64(3), str(4)
memory usage: 417.3 MB


In [6]:
df.head()

,id,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,trip_duration
0,id2875421,2,2016-03-14 17:24:55,2016-03-14 17:32:30,1,-73.982155,40.767937,-73.964630,40.765602,N,455
1,id2377394,1,2016-06-12 00:43:35,2016-06-12 00:54:38,1,-73.980415,40.738564,-73.999481,40.731152,N,663
2,id3858529,2,2016-01-19 11:35:24,2016-01-19 12:10:48,1,-73.979027,40.763939,-74.005333,40.710087,N,2124
3,id3504673,2,2016-04-06 19:32:31,2016-04-06 19:39:40,1,-74.010040,40.719971,-74.012268,40.706718,N,429
4,id2181028,2,2016-03-26 13:30:55,2016-03-26 13:38:10,1,-73.973053,40.793209,-73.972923,40.782520,N,435


### Data Quality Understanding

In [7]:
df.isnull().sum()

id                    0
vendor_id             0
pickup_datetime       0
dropoff_datetime      0
passenger_count       0
pickup_longitude      0
pickup_latitude       0
dropoff_longitude     0
dropoff_latitude      0
store_and_fwd_flag    0
trip_duration         0
dtype: int64

In [8]:
df.duplicated().sum()

np.int64(0)

In [ ]:
df['id'].duplicated().sum()

In [ ]:
df['trip_duration'].describe()

In [ ]:
df["trip_duration"].quantile([0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99, 0.999])

In [ ]:
df['trip_duration'].nsmallest(20)

In [ ]:
df['trip_duration'].nlargest(20)

In [ ]:
df[[
    'pickup_longitude',
    'pickup_latitude',
    'dropoff_longitude',
    'dropoff_latitude'
]].describe()

In [ ]:
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])
df['dropoff_datetime'] = pd.to_datetime(df['dropoff_datetime'])

In [ ]:
calculated_duration = (df['dropoff_datetime'] - df['pickup_datetime']).dt.total_seconds()

In [ ]:
(df['trip_duration'] != calculated_duration).sum()

In [ ]:
for min in [1, 2, 5, 60, 120, 180, 300]:
  count = (df['trip_duration'] > min * 60).sum()
  print(f'{min:>3} minutes: {count:,}')

In [ ]:
for sec in [1, 2, 5, 10, 30, 60]:
  count = (df['trip_duration'] < sec).sum()
  print(f'{sec:>2} seconds: {count:,}')

### Haversine Formula

In [ ]:
lat1 = np.radians(df['pickup_latitude'])
lon1 = np.radians(df['pickup_longitude'])
lat2 = np.radians(df['dropoff_latitude'])
lon2 = np.radians(df['dropoff_longitude'])

dlat = lat2 - lat1
dlon = lon2 - lon1

a = (np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2)

c = 2 * np.arcsin(np.sqrt(a))

df["distance_km"] = 6371 * c

In [ ]:
df['distance_km'].describe()

In [ ]:
df['distance_km'].quantile([0.001, 0.01, 0.25, 0.50, 0.75, 0.99, 0.999, 0.9999])

In [ ]:
df.loc[df['distance_km'].nlargest(10).index,[
        'id',
        'pickup_datetime',
        'dropoff_datetime',
        'pickup_longitude',
        'pickup_latitude',
        'dropoff_longitude',
        'dropoff_latitude',
        'trip_duration'
]].sort_values('trip_duration')

In [ ]:
df.loc[df['distance_km'].nsmallest(10).index,[
        'id',
        'pickup_datetime',
        'dropoff_datetime',
        'pickup_longitude',
        'pickup_latitude',
        'dropoff_longitude',
        'dropoff_latitude',
        'trip_duration'
]].sort_values('trip_duration')

In [ ]:
for dist in [30, 50, 100, 200, 500]:
  count = (df['distance_km'] > dist).sum()
  percentage = count / len(df) * 100
  print(f'> {dist} km: {count:,} {percentage:.4f}')

In [ ]:
df.loc[df['distance_km'] > 100, [
        'pickup_longitude',
        'pickup_latitude',
        'dropoff_longitude',
        'dropoff_latitude',
        'trip_duration',
        'distance_km'
]].sort_values('trip_duration')

In [ ]:
df['avg_speed_kmh'] = (df['distance_km'] / (df['trip_duration'] / 3600))

In [ ]:
df['avg_speed_kmh'].describe(percentiles=[0.001, 0.01, 0.5, 0.99, 0.999])

In [ ]:
df.nlargest(20, 'avg_speed_kmh')[[
        'distance_km',
        'trip_duration',
        'avg_speed_kmh',
        'pickup_longitude',
        'pickup_latitude',
        'dropoff_longitude',
        'dropoff_latitude'
]].sort_values('trip_duration')

In [ ]:
for speed in [80, 100, 120, 150, 200]:
  count = (df['avg_speed_kmh'] > speed).sum()
  percentage = count / len(df) * 100
  print(f'> {speed} kmh: {count:,} {percentage:.4f}')

In [ ]:
df[
    (df['avg_speed_kmh'] > 100) &
    (df['avg_speed_kmh'] <= 150)
][[
        'distance_km',
        'trip_duration',
        'avg_speed_kmh',
        'pickup_longitude',
        'pickup_latitude',
        'dropoff_longitude',
        'dropoff_latitude'
 ]].sort_values('avg_speed_kmh', ascending=False).head(20)

In [ ]:
for speed in [0.1, 0.5, 1, 2, 3, 5]:
  count = (df['avg_speed_kmh'] < speed).sum()
  percentage = count / len(df) * 100
  print(f'< {speed} kmh: {count:,} {percentage:.4f}')

In [ ]:
df.nsmallest(20, 'avg_speed_kmh')[[
        'distance_km',
        'trip_duration',
        'avg_speed_kmh',
        'pickup_longitude',
        'pickup_latitude',
        'dropoff_longitude',
        'dropoff_latitude'
]].sort_values('trip_duration')

In [ ]:
speed_invalid = df['avg_speed_kmh'] > 120

print('Invalid speed rows: ', speed_invalid.sum())
print('percentage: ', speed_invalid.mean() * 100)

In [ ]:
df = df[~speed_invalid].copy()

In [ ]:
print('New shape: ', df.shape)
print('Max speed: ', df['avg_speed_kmh'].max())

In [ ]:
for sec in [3600, 7200, 10800, 18000, 36000, 86400]:
  count = (df['trip_duration'] > sec).sum()
  percentage = count / len(df) * 100
  print(
      f'> {sec / 3600:.0f} hour(s)'
      f'{count:,} {percentage:0.4f}'
  )

In [ ]:
df.nlargest(20, 'trip_duration')[[
        'distance_km',
        'trip_duration',
        'avg_speed_kmh',
        'pickup_longitude',
        'pickup_latitude',
        'dropoff_longitude',
        'dropoff_latitude'
]]

In [ ]:
df[
    (df['trip_duration'] >= 86000) &
    (df['trip_duration'] < 86400)
]['trip_duration'].describe()

In [ ]:
duration_invalid = df['trip_duration'] >= 86000

print('Invalid duration rows: ', duration_invalid.sum())

combined_invalid = speed_invalid | duration_invalid

print('Combined invalid rows: ', combined_invalid.sum())
print('Percentage: ', combined_invalid.mean())

In [ ]:
df = df[~duration_invalid].copy()

In [ ]:
print('Remaining rows: ', len(df))
print('Max duration: ', df['trip_duration'].max())
print('Max duration hour:', df['trip_duration'].max() / 3600)

In [ ]:
df.isnull().sum().sum()
df.duplicated().sum()

### Timestamp Feature Engineering

In [ ]:
df['pickup_datetime'].dtype
df['dropoff_datetime'].dtype

In [ ]:
print(df['pickup_datetime'].min(), 'to', df['pickup_datetime'].max())
print(df['dropoff_datetime'].min(), 'to', df['dropoff_datetime'].max())

In [ ]:
df['pickup_hour'] = df['pickup_datetime'].dt.hour
df['pickup_day'] = df['pickup_datetime'].dt.day
df['pickup_dayofweek'] = df['pickup_datetime'].dt.dayofweek
df['pickup_month'] = df['pickup_datetime'].dt.month

In [ ]:
df['is_weekend'] = (df['pickup_dayofweek'] >= 5).astype(int)

In [ ]:
df['is_rush_hour'] = (
    (df['pickup_dayofweek'] < 5) & (
        df['pickup_hour'].between(7, 9) | df['pickup_hour'].between(16, 19))
).astype(int)

### EDA: Feature Engineering

In [ ]:
hour_duration = (df.groupby('pickup_hour')['trip_duration'].agg(['count', 'mean', 'median']))
hour_duration

In [ ]:
day_duration = (df.groupby('pickup_dayofweek')['trip_duration'].agg(['count', 'mean', 'median']))
day_duration

In [ ]:
df[['distance_km', 'trip_duration']].corr()

In [ ]:
df[['pickup_latitude', 'pickup_longitude', 'trip_duration']].corr()

In [ ]:
df[df['distance_km'] < 0.5][['distance_km', 'trip_duration', 'avg_speed_kmh']].describe()

In [ ]:
df[df['distance_km'] > 30][['distance_km', 'trip_duration', 'avg_speed_kmh']].describe()

In [ ]:
df[df['trip_duration'] > 36000][[
        'pickup_datetime',
        'dropoff_datetime',
        'distance_km',
        'trip_duration',
        'avg_speed_kmh'
]].head(20)

In [ ]:
df[df['trip_duration'] > 36000]['avg_speed_kmh'].describe()

In [ ]:
invalid_duration = df['trip_duration'] > 36000

df = df[~invalid_duration].copy()
print('Max duration: ', df['trip_duration'].max())

### Feature Engineering

### Manhattan Distance

In [ ]:
lat_diff = np.radians(df['dropoff_latitude'] - df['pickup_latitude'])
lon_diff = np.radians(df['dropoff_longitude'] - df['pickup_longitude'])

mean_lat = np.radians((df['dropoff_latitude'] + df['pickup_latitude']) / 2)

lat_distance_km = 6371 * np.abs(lat_diff)

lon_distance_km = (6371 * np.abs(lon_diff) * np.cos(mean_lat))

df['manhattan_distance_km'] = lat_distance_km + lon_distance_km

In [ ]:
df[['distance_km', 'manhattan_distance_km', 'trip_duration']].describe()

In [ ]:
df[['distance_km', 'manhattan_distance_km', 'trip_duration']].corr()

### Bearing / Direction

In [ ]:
x = np.sin(dlon) * np.cos(lat2)

y = (np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon))

df["bearing"] = (np.degrees(np.arctan2(x, y)) + 360) % 360

In [ ]:
df['bearing'].describe()

In [ ]:
df[['bearing', 'trip_duration']].corr()

In [ ]:
df['bearing_sin'] = np.sin(np.radians(df['bearing']))
df['bearing_cos'] = np.cos(np.radians(df['bearing']))

### Geographic Clustering: K-Means

In [ ]:
coords = df[['pickup_latitude', 'pickup_longitude']]
coords_sample = coords.sample(n = 100_000, random_state=1)

In [ ]:
k_values = [10, 20, 30, 40, 50]

inertias = []

for k in k_values:
  kmeans = KMeans(n_clusters=k, random_state=1, n_init=10)
  kmeans.fit(coords_sample)
  inertias.append(kmeans.inertia_)

In [ ]:
pd.DataFrame({
    'k': k_values,
    'inertia': inertias
})

### Cyclic Time Encoding

In [ ]:
df['hour_sin'] = np.sin(2 * np.pi * df['pickup_hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['pickup_hour'] / 24)

In [ ]:
df[['pickup_hour', 'hour_sin', 'hour_cos']].drop_duplicates().sort_values('pickup_hour')

In [ ]:
df['dayofweek_sin'] = np.sin(2 * np.pi * df['pickup_dayofweek'] / 7)
df['dayofweek_cos'] = np.cos(2 * np.pi * df['pickup_dayofweek'] / 7)

In [ ]:
df[['pickup_dayofweek', 'dayofweek_sin', 'dayofweek_cos']].drop_duplicates().sort_values('pickup_dayofweek')

In [ ]:
df['route_detour_ratio'] = np.where(
    df['distance_km'] > 0,
    df['manhattan_distance_km'] / (df['distance_km'] + 1e-6),
    0
)

In [ ]:
df['route_detour_ratio'].describe()

In [ ]:
df[['route_detour_ratio', 'trip_duration']].corr()

### Model Preparation

In [ ]:
model_df = df.drop(columns=['id', 'dropoff_datetime', 'avg_speed_kmh', 'bearing']).copy()

In [ ]:
X = model_df.drop(columns='trip_duration')

In [ ]:
y = model_df['trip_duration']

In [ ]:
print(X.shape, y.shape)

In [ ]:
X.dtypes

### Chronological Splitting

In [ ]:
model_df = model_df.sort_values('pickup_datetime').reset_index(drop=True)

In [ ]:
n = len(model_df)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = model_df.iloc[:train_end].copy()
val_df = model_df.iloc[train_end:val_end].copy()
test_df = model_df.iloc[val_end:].copy()

In [ ]:
print('train', train_df['pickup_datetime'].min(), 'to', train_df['pickup_datetime'].max())
print('val', val_df['pickup_datetime'].min(), 'to', val_df['pickup_datetime'].max())
print('test', test_df['pickup_datetime'].min(), 'to', test_df['pickup_datetime'].max())

In [ ]:
target = 'trip_duration'

X_train = train_df.drop(columns=[target, 'pickup_datetime'])
y_train = train_df[target]

X_val = val_df.drop(columns=[target, 'pickup_datetime'])
y_val = val_df[target]

X_test = test_df.drop(columns=[target, 'pickup_datetime'])
y_test = test_df[target]

In [ ]:
pickup_kmeans = KMeans(n_clusters=50, random_state=1, n_init=10)
dropoff_kmeans = KMeans(n_clusters=50, random_state=1, n_init=10)

In [ ]:
pickup_kmeans.fit(X_train[['pickup_latitude', 'pickup_longitude']])
dropoff_kmeans.fit(X_train[['dropoff_latitude', 'dropoff_longitude']])

In [ ]:
X_train['pickup_zone'] = pickup_kmeans.predict(X_train[['pickup_latitude', 'pickup_longitude']])
X_train['dropoff_zone'] = dropoff_kmeans.predict(X_train[['dropoff_latitude', 'dropoff_longitude']])

X_val['pickup_zone'] = pickup_kmeans.predict(X_val[['pickup_latitude', 'pickup_longitude']])
X_val['dropoff_zone'] = dropoff_kmeans.predict(X_val[['dropoff_latitude', 'dropoff_longitude']])

X_test['pickup_zone'] = pickup_kmeans.predict(X_test[['pickup_latitude', 'pickup_longitude']])
X_test['dropoff_zone'] = dropoff_kmeans.predict(X_test[['dropoff_latitude', 'dropoff_longitude']])

In [ ]:
print(X_train.shape, X_test.shape, X_val.shape)

In [ ]:
numeric_features = [
 'passenger_count',
 'pickup_longitude',
 'pickup_latitude',
 'dropoff_longitude',
 'dropoff_latitude',
 'distance_km',
 'pickup_hour',
 'pickup_day',
 'pickup_dayofweek',
 'pickup_month',
 'is_weekend',
 'is_rush_hour',
 'manhattan_distance_km',
 'hour_sin',
 'hour_cos',
 'dayofweek_sin',
 'dayofweek_cos',
 'route_detour_ratio',
 'bearing_sin',
 'bearing_cos']

categorical_features = [
     'vendor_id',
     'store_and_fwd_flag',
     'pickup_zone',
     'dropoff_zone'
 ]

### Preprocessing Transformer

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

### Linear Regression Model

In [ ]:
if RUN_OLD_MODELS:
  linear_model = Pipeline(
      steps=[
          ('preprocessor', preprocessor),
          ('model', LinearRegression())
      ]
  )

In [ ]:
if RUN_OLD_MODELS:
  linear_model.fit(X_train, y_train)

In [ ]:
# y_val_pred = linear_model.predict(X_val)

In [ ]:
# mae = round(mean_absolute_error(y_val, y_val_pred), 3)
# rmse = round(np.sqrt(mean_squared_error(y_val, y_val_pred)), 3)
# r2 = round(r2_score(y_val, y_val_pred), 3)

# print('Linear regression:')
# print('MAE: ', mae)
# print('RMSE: ', rmse)
# print('R2 score: ', r2)

In [ ]:
# print((y_val_pred < 0).sum())
# print(y_val_pred.min())
# print(y_val_pred.max())

### Ridge Regression

In [ ]:
if RUN_OLD_MODELS:
  ridge_model = Pipeline(
      steps=[
          ('preprocessor', preprocessor),
          ('model', Ridge(alpha=1.0))
      ]
  )

In [ ]:
if RUN_OLD_MODELS:
  ridge_model.fit(X_train, y_train)

In [ ]:
# y_val_pred_ridge = ridge_model.predict(X_val)

In [ ]:
# mae_ridge = round(mean_absolute_error(y_val, y_val_pred_ridge), 3)
# rmse_ridge = round(np.sqrt(mean_squared_error(y_val, y_val_pred_ridge)), 3)
# r2_ridge = round(r2_score(y_val, y_val_pred_ridge), 3)

# print('Ridge regression:')
# print('MAE: ', mae_ridge)
# print('RMSE: ', rmse_ridge)
# print('R2 score: ', r2_ridge)

### Random Forest

In [ ]:
X_train_rf = X_train.sample(n=200_000, random_state=1)

In [ ]:
y_train_rf = y_train.loc[X_train_rf.index]

In [ ]:
if RUN_OLD_MODELS:
  rf_pipeline = Pipeline(
      steps=[
          ('preprocessor', preprocessor),
          ('model', RandomForestRegressor(
              n_estimators=25,
              max_depth=15,
              min_samples_leaf=2,
              random_state=1,
              n_jobs=-1
          ))
      ]
  )

In [ ]:
if RUN_OLD_MODELS:
  rf_pipeline.fit(X_train_rf, y_train_rf)

In [ ]:
# y_val_pred_rf_pipeline = rf_pipeline.predict(X_val)

In [ ]:
# mae_rf_pipeline = round(mean_absolute_error(y_val, y_val_pred_rf_pipeline), 3)
# rmse_rf_pipeline = round(np.sqrt(mean_squared_error(y_val, y_val_pred_rf_pipeline)), 3)
# r2_rf_pipeline = round(r2_score(y_val, y_val_pred_rf_pipeline), 3)

# print('Random forest:')
# print('MAE: ', mae_rf_pipeline)
# print('RMSE: ', rmse_rf_pipeline)
# print('R2 score: ', r2_rf_pipeline)

### Hist Gradient Boosting Regressor

In [ ]:
hgb_preprocessor = ColumnTransformer(
      transformers=[
          ('num', StandardScaler(), numeric_features),
          ('cat', OneHotEncoder(
              handle_unknown='ignore',
              sparse_output=False,
              dtype=np.float32
          ), categorical_features)
      ]
  )

In [ ]:
if RUN_OLD_MODELS:
  hgb_model = Pipeline(
      steps=[
          ('preprocessor', hgb_preprocessor),
          ('model', HistGradientBoostingRegressor(
            max_iter=100,
            learning_rate=0.1,
            max_leaf_nodes=31,
            random_state=1
          ))
      ]
  )

In [ ]:
if RUN_OLD_MODELS:
  hgb_model.fit(X_train, y_train)

In [ ]:
# y_val_pred_hgb = hgb_model.predict(X_val)

In [ ]:
# mae_hgb = round(mean_absolute_error(y_val, y_val_pred_hgb), 3)
# rmse_hgb = round(np.sqrt(mean_squared_error(y_val, y_val_pred_hgb)), 3)
# r2_hgb = round(r2_score(y_val, y_val_pred_hgb), 3)

# print('Hist gradient boosting:')
# print('MAE: ', mae_hgb)
# print('RMSE: ', rmse_hgb)
# print('R2 score: ', r2_hgb)

### HGB: Hyperparameter tuning

In [ ]:
if RUN_OLD_MODELS:
  hgb_model_200 = Pipeline(
      steps=[
          ('preprocessor', hgb_preprocessor),
          ('model', HistGradientBoostingRegressor(
            max_iter=200,
            learning_rate=0.1,
            max_leaf_nodes=31,
            random_state=1
          ))
      ]
  )

In [ ]:
if RUN_OLD_MODELS:
  hgb_model_200.fit(X_train, y_train)

In [ ]:
# y_val_pred_hgb_200 = hgb_model_200.predict(X_val)

In [ ]:
# mae_hgb_200 = round(mean_absolute_error(y_val, y_val_pred_hgb_200), 3)
# rmse_hgb_200 = round(np.sqrt(mean_squared_error(y_val, y_val_pred_hgb_200)), 3)
# r2_hgb_200 = round(r2_score(y_val, y_val_pred_hgb_200), 3)

# print('Hist gradient boosting(200):')
# print('MAE: ', mae_hgb_200)
# print('RMSE: ', rmse_hgb_200)
# print('R2 score: ', r2_hgb_200)

In [ ]:
if RUN_OLD_MODELS:
  hgb_model_300 = Pipeline(
      steps=[
          ('preprocessor', hgb_preprocessor),
          ('model', HistGradientBoostingRegressor(
            max_iter=300,
            learning_rate=0.1,
            max_leaf_nodes=31,
            random_state=1
          ))
      ]
  )

In [ ]:
if RUN_OLD_MODELS:
  hgb_model_300.fit(X_train, y_train)

In [ ]:
# y_val_pred_hgb_300 = hgb_model_300.predict(X_val)

In [ ]:
# mae_hgb_300 = round(mean_absolute_error(y_val, y_val_pred_hgb_300), 3)
# rmse_hgb_300 = round(np.sqrt(mean_squared_error(y_val, y_val_pred_hgb_300)), 3)
# r2_hgb_300 = round(r2_score(y_val, y_val_pred_hgb_300), 3)

# print('Hist gradient boosting(300):')
# print('MAE: ', mae_hgb_300)
# print('RMSE: ', rmse_hgb_300)
# print('R2 score: ', r2_hgb_300)

In [ ]:
if RUN_OLD_MODELS:
  hgb_300_lr005 = Pipeline(
      steps=[
          ('preprocessor', hgb_preprocessor),
          ('model', HistGradientBoostingRegressor(
            max_iter=300,
            learning_rate=0.05,
            max_leaf_nodes=31,
            random_state=1
          ))
      ]
  )

In [ ]:
if RUN_OLD_MODELS:
  hgb_300_lr005.fit(X_train, y_train)

In [ ]:
# y_val_pred_hgb_300_lr005 = hgb_300_lr005.predict(X_val)

In [ ]:
# mae_hgb_300_lr005 = round(mean_absolute_error(y_val, y_val_pred_hgb_300_lr005), 3)
# rmse_hgb_300_lr005 = round(np.sqrt(mean_squared_error(y_val, y_val_pred_hgb_300_lr005)), 3)
# r2_hgb_300_lr005 = round(r2_score(y_val, y_val_pred_hgb_300_lr005), 3)

# print('Hist gradient boosting(300_lr005):')
# print('MAE: ', mae_hgb_300_lr005)
# print('RMSE: ', rmse_hgb_300_lr005)
# print('R2 score: ', r2_hgb_300_lr005)

In [ ]:
if RUN_OLD_MODELS:
  hgb_model_leaf15 = Pipeline(
      steps=[
          ('preprocessor', hgb_preprocessor),
          ('model', HistGradientBoostingRegressor(
            max_iter=300,
            learning_rate=0.1,
            max_leaf_nodes=15,
            random_state=1
          ))
      ]
  )

In [ ]:
if RUN_OLD_MODELS:
  hgb_model_leaf15.fit(X_train, y_train)

In [ ]:
# y_val_pred_hgb_leaf15 = hgb_model_leaf15.predict(X_val)

In [ ]:
# mae_hgb_leaf15 = round(mean_absolute_error(y_val, y_val_pred_hgb_leaf15), 3)
# rmse_hgb_leaf15 = round(np.sqrt(mean_squared_error(y_val, y_val_pred_hgb_leaf15)), 3)
# r2_hgb_leaf15 = round(r2_score(y_val, y_val_pred_hgb_leaf15), 3)

# print('Hist gradient boosting(leaf15):')
# print('MAE: ', mae_hgb_leaf15)
# print('RMSE: ', rmse_hgb_leaf15)
# print('R2 score: ', r2_hgb_leaf15)

In [ ]:
if RUN_OLD_MODELS:
  hgb_model_leaf63 = Pipeline(
      steps=[
          ('preprocessor', hgb_preprocessor),
          ('model', HistGradientBoostingRegressor(
            max_iter=300,
            learning_rate=0.1,
            max_leaf_nodes=63,
            random_state=1
          ))
      ]
  )

In [ ]:
if RUN_OLD_MODELS:
  hgb_model_leaf63.fit(X_train, y_train)

In [ ]:
# y_val_pred_hgb_leaf63 = hgb_model_leaf63.predict(X_val)

In [ ]:
# mae_hgb_leaf63 = round(mean_absolute_error(y_val, y_val_pred_hgb_leaf63), 3)
# rmse_hgb_leaf63 = round(np.sqrt(mean_squared_error(y_val, y_val_pred_hgb_leaf63)), 3)
# r2_hgb_leaf63 = round(r2_score(y_val, y_val_pred_hgb_leaf63), 3)

# print('Hist gradient boosting(leaf63):')
# print('MAE: ', mae_hgb_leaf63)
# print('RMSE: ', rmse_hgb_leaf63)
# print('R2 score: ', r2_hgb_leaf63)

In [ ]:
if RUN_OLD_MODELS:
  hgb_model_minleaf50 = Pipeline(
      steps=[
          ('preprocessor', hgb_preprocessor),
          ('model', HistGradientBoostingRegressor(
            max_iter=300,
            learning_rate=0.1,
            max_leaf_nodes=63,
            min_samples_leaf=50,
            random_state=1
          ))
      ]
  )

In [ ]:
if RUN_OLD_MODELS:
  hgb_model_minleaf50.fit(X_train, y_train)

In [ ]:
# y_val_pred_hgb_minleaf50 = hgb_model_minleaf50.predict(X_val)

In [ ]:
# mae_hgb_minleaf50 = round(mean_absolute_error(y_val, y_val_pred_hgb_minleaf50), 3)
# rmse_hgb_minleaf50 = round(np.sqrt(mean_squared_error(y_val, y_val_pred_hgb_minleaf50)), 3)
# r2_hgb_minleaf50 = round(r2_score(y_val, y_val_pred_hgb_minleaf50), 3)

# print('Hist gradient boosting(minleaf50):')
# print('MAE: ', mae_hgb_minleaf50)
# print('RMSE: ', rmse_hgb_minleaf50)
# print('R2 score: ', r2_hgb_minleaf50)

In [ ]:
if RUN_OLD_MODELS:
  hgb_model_l2 = Pipeline(
      steps=[
          ('preprocessor', hgb_preprocessor),
          ('model', HistGradientBoostingRegressor(
            max_iter=300,
            learning_rate=0.1,
            max_leaf_nodes=63,
            min_samples_leaf=50,
            l2_regularization=1.0,
            random_state=1
          ))
      ]
  )

In [ ]:
if RUN_OLD_MODELS:
  hgb_model_l2.fit(X_train, y_train)

In [ ]:
# y_val_pred_hgb_l2 = hgb_model_l2.predict(X_val)

In [ ]:
# mae_hgb_l2 = round(mean_absolute_error(y_val, y_val_pred_hgb_l2), 3)
# rmse_hgb_l2 = round(np.sqrt(mean_squared_error(y_val, y_val_pred_hgb_l2)), 3)
# r2_hgb_l2 = round(r2_score(y_val, y_val_pred_hgb_l2), 3)

# print('Hist gradient boosting(l2):')
# print('MAE: ', mae_hgb_l2)
# print('RMSE: ', rmse_hgb_l2)
# print('R2 score: ', r2_hgb_l2)

In [ ]:
if RUN_OLD_MODELS:
  hgb_model_500 = Pipeline(
      steps=[
          ('preprocessor', hgb_preprocessor),
          ('model', HistGradientBoostingRegressor(
            max_iter=500,
            learning_rate=0.1,
            max_leaf_nodes=63,
            min_samples_leaf=50,
            random_state=1
          ))
      ]
  )

In [ ]:
if RUN_OLD_MODELS:
  hgb_model_500.fit(X_train, y_train)

In [ ]:
# y_val_pred_hgb_500 = hgb_model_500.predict(X_val)

In [ ]:
# mae_hgb_500 = round(mean_absolute_error(y_val, y_val_pred_hgb_500), 3)
# rmse_hgb_500 = round(np.sqrt(mean_squared_error(y_val, y_val_pred_hgb_500)), 3)
# r2_hgb_500 = round(r2_score(y_val, y_val_pred_hgb_500), 3)

# print('Hist gradient boosting(500):')
# print('MAE: ', mae_hgb_500)
# print('RMSE: ', rmse_hgb_500)
# print('R2 score: ', r2_hgb_500)

In [ ]:
RUN_FINAL_HGB = False

In [ ]:
if RUN_FINAL_HGB:
  hgb_model_700 = Pipeline(
      steps=[
          ('preprocessor', hgb_preprocessor),
          ('model', HistGradientBoostingRegressor(
            max_iter=700,
            learning_rate=0.1,
            max_leaf_nodes=63,
            min_samples_leaf=50,
            l2_regularization=1.0,
            random_state=1
          ))
      ]
  )

In [ ]:
if RUN_FINAL_HGB:
  hgb_model_700.fit(X_train, y_train)

In [ ]:
# y_val_pred_hgb_700 = hgb_model_700.predict(X_val)

In [ ]:
# mae_hgb_700 = round(mean_absolute_error(y_val, y_val_pred_hgb_700), 3)
# rmse_hgb_700 = round(np.sqrt(mean_squared_error(y_val, y_val_pred_hgb_700)), 3)
# r2_hgb_700 = round(r2_score(y_val, y_val_pred_hgb_700), 3)

# print('Hist gradient boosting(700):')
# print('MAE: ', mae_hgb_700)
# print('RMSE: ', rmse_hgb_700)
# print('R2 score: ', r2_hgb_700)

### Lasso Model

In [ ]:
if RUN_OLD_MODELS:
  lasso_model = Pipeline(
      steps=[
          ('preprocessor', preprocessor),
          ('model', Lasso(
              alpha=0.01,
              max_iter=10000,
              random_state=1
          ))
      ]
  )

In [ ]:
if RUN_OLD_MODELS:
  lasso_model.fit(X_train, y_train)

In [ ]:
# y_val_pred_lasso = lasso_model.predict(X_val)

In [ ]:
# mae_lasso = round(mean_absolute_error(y_val, y_val_pred_lasso), 3)
# rmse_lasso = round(np.sqrt(mean_squared_error(y_val, y_val_pred_lasso)), 3)
# r2_lasso = round(r2_score(y_val, y_val_pred_lasso), 3)

# print('Lasso:')
# print('MAE: ', mae_lasso)
# print('RMSE: ', rmse_lasso)
# print('R2 score: ', r2_lasso)

### ElasticNet Model

In [ ]:
if RUN_OLD_MODELS:
  elasticnet_model = Pipeline(
      steps=[
          ('preprocessor', preprocessor),
          ('model', ElasticNet(
              alpha=0.01,
              l1_ratio=0.5,
              max_iter=10000,
              random_state=1
          ))
      ]
  )

In [ ]:
if RUN_OLD_MODELS:
  elasticnet_model.fit(X_train, y_train)

In [ ]:
# y_val_pred_eln = elasticnet_model.predict(X_val)

In [ ]:
# mae_eln = round(mean_absolute_error(y_val, y_val_pred_eln), 3)
# rmse_eln = round(np.sqrt(mean_squared_error(y_val, y_val_pred_eln)), 3)
# r2_eln = round(r2_score(y_val, y_val_pred_eln), 3)

# print('ElasticNet:')
# print('MAE: ', mae_eln)
# print('RMSE: ', rmse_eln)
# print('R2 score: ', r2_eln)

### XGBregressor Model

In [ ]:
if RUN_OLD_MODELS:
  xgb_model = Pipeline(
        steps=[
            ('preprocessor', preprocessor),
            ('model', XGBRegressor(
                n_estimators=100,
                learning_rate=0.1,
                max_depth=6,
                subsample=1.0,
                colsample_bytree=1.0,
                objective='reg:squarederror',
                random_state=1,
                n_jobs=-1
            ))
        ]
    )

In [ ]:
if RUN_OLD_MODELS:
  xgb_model.fit(X_train, y_train)

In [ ]:
# y_val_pred_xgb = xgb_model.predict(X_val)

In [ ]:
# mae_xgb = round(mean_absolute_error(y_val, y_val_pred_xgb), 3)
# rmse_xgb = round(np.sqrt(mean_squared_error(y_val, y_val_pred_xgb)), 3)
# r2_xgb = round(r2_score(y_val, y_val_pred_xgb), 3)

# print('XGBregressor:')
# print('MAE: ', mae_xgb)
# print('RMSE: ', rmse_xgb)
# print('R2 score: ', r2_xgb)

### Hyperparametr Tuning

In [ ]:
if RUN_OLD_MODELS:
  xgb_tuning_result = []

  for max_depth in [4, 6, 8]:
    for min_child_weight in [1, 5, 10]:
      model = Pipeline(
        steps=[
            ('preprocessor', preprocessor),
            ('model', XGBRegressor(
                n_estimators=200,
                learning_rate=0.1,
                max_depth=max_depth,
                min_child_weight=min_child_weight,
                subsample=1.0,
                colsample_bytree=1.0,
                objective='reg:squarederror',
                random_state=1,
                n_jobs=-1
            ))
        ]
      )

      model.fit(X_train, y_train)

      y_val_pred_model = model.predict(X_val)

      mae_model = round(mean_absolute_error(y_val, y_val_pred_model), 3)
      rmse_model = round(np.sqrt(mean_squared_error(y_val, y_val_pred_model)), 3)
      r2_model = round(r2_score(y_val, y_val_pred_model), 3)

      xgb_tuning_result.append({
          'max_depth': max_depth,
          'min_child_weight': min_child_weight,
          'MAE': mae_model,
          'RMSE': rmse_model,
          'R2': r2_model
      })

  xgb_tuning_result_df = pd.DataFrame(xgb_tuning_result)
  xgb_tuning_result_df.sort_values('MAE')

In [ ]:
if RUN_OLD_MODELS:
  xgb_lr_result = []

  for learning_rate, n_estimators in [
      (0.10, 200), (0.10, 300), (0.10, 500), (0.05, 300), (0.05, 500), (0.05, 700), (0.03, 700)
  ]:
    model = Pipeline(
            steps=[
                ('preprocessor', preprocessor),
                ('model', XGBRegressor(
                    n_estimators=n_estimators,
                    learning_rate=learning_rate,
                    max_depth=8,
                    min_child_weight=10,
                    subsample=1.0,
                    colsample_bytree=1.0,
                    objective='reg:squarederror',
                    random_state=1,
                    n_jobs=-1
                ))
            ]
          )

    model.fit(X_train, y_train)

    y_val_pred_model = model.predict(X_val)

    mae_model = round(mean_absolute_error(y_val, y_val_pred_model), 3)
    rmse_model = round(np.sqrt(mean_squared_error(y_val, y_val_pred_model)), 3)
    r2_model = round(r2_score(y_val, y_val_pred_model), 3)

    xgb_lr_result.append({
              'learning_rate': learning_rate,
              'n_estimators': n_estimators,
              'MAE': mae_model,
              'RMSE': rmse_model,
              'R2': r2_model
          })

  xgb_lr_result_df = pd.DataFrame(xgb_lr_result)
  xgb_lr_result_df.sort_values('MAE')

In [ ]:
if RUN_OLD_MODELS:
  xgb_sampling_result = []

  for subsample, colsample_bytree in [
      (1.0, 1.0), (0.8, 1.0), (1.0, 0.8), (0.8, 0.8)
  ]:
    model = Pipeline(
            steps=[
                ('preprocessor', preprocessor),
                ('model', XGBRegressor(
                    n_estimators=300,
                    learning_rate=0.1,
                    max_depth=8,
                    min_child_weight=10,
                    subsample=subsample,
                    colsample_bytree=colsample_bytree,
                    objective='reg:squarederror',
                    random_state=1,
                    n_jobs=-1
                ))
            ]
          )

    model.fit(X_train, y_train)

    y_val_pred_model = model.predict(X_val)

    mae_model = round(mean_absolute_error(y_val, y_val_pred_model), 3)
    rmse_model = round(np.sqrt(mean_squared_error(y_val, y_val_pred_model)), 3)
    r2_model = round(r2_score(y_val, y_val_pred_model), 3)

    xgb_sampling_result.append({
              'subsample': subsample,
              'colsample_bytree': colsample_bytree,
              'MAE': mae_model,
              'RMSE': rmse_model,
              'R2': r2_model
          })

  xgb_sampling_result_df = pd.DataFrame(xgb_sampling_result)
  xgb_sampling_result_df.sort_values('MAE')

In [ ]:
if RUN_OLD_MODELS:
  xgb_gamma_result = []

  for gamma in [0, 0.1, 0.5, 1]:
    model = Pipeline(
              steps=[
                  ('preprocessor', preprocessor),
                  ('model', XGBRegressor(
                      n_estimators=300,
                      learning_rate=0.1,
                      max_depth=8,
                      min_child_weight=10,
                      subsample=1.0,
                      colsample_bytree=1.0,
                      gamma=gamma,
                      objective='reg:squarederror',
                      random_state=1,
                      n_jobs=-1
                  ))
              ]
            )

    model.fit(X_train, y_train)

    y_val_pred_model = model.predict(X_val)

    mae_model = round(mean_absolute_error(y_val, y_val_pred_model), 3)
    rmse_model = round(np.sqrt(mean_squared_error(y_val, y_val_pred_model)), 3)
    r2_model = round(r2_score(y_val, y_val_pred_model), 3)

    xgb_gamma_result.append({
                'gamma': gamma,
                'MAE': mae_model,
                'RMSE': rmse_model,
                'R2': r2_model
            })

  xgb_gamma_result_df = pd.DataFrame(xgb_gamma_result)
  xgb_gamma_result_df.sort_values('MAE')

In [ ]:
result = pd.DataFrame([
    {
        'Model': 'Linear Regression',
        'MAE(sec)': 283.994,
        'RMSE(sec)': 465.318,
        'R2': 0.616
    },

    {
        'Model': 'Ridge Regression',
        'MAE(sec)': 283.977,
        'RMSE(sec)': 465.280,
        'R2': 0.616
    },

    {
        'Model': 'Lasso Regression',
        'MAE(sec)': 283.113,
        'RMSE(sec)': 465.682,
        'R2': 0.616
    },

    {
        'Model': 'ElasticNet',
        'MAE(sec)': 290.575,
        'RMSE(sec)': 472.687,
        'R2': 0.604
    },

    {
        'Model': 'Random Forest',
        'MAE(sec)': 220.318,
        'RMSE(sec)': 404.979,
        'R2': 0.709
    },

    {
        'Model': 'HistGradienBoosting(Initial)',
        'MAE(sec)': 212.685,
        'RMSE(sec)': 392.965,
        'R2': 0.726
    },

    {
        'Model': 'HistGradienBoosting(Tuned)',
        'MAE(sec)': 205.636,
        'RMSE(sec)': 378.647,
        'R2': 0.746
    },

    {
        'Model': 'XGBoost(Tuned)',
        'MAE(sec)': 202.525,
        'RMSE(sec)': 379.612,
        'R2': 0.745
    },
])
result

### Final Evaluation

In [ ]:
final_xgb_model = Pipeline(
                steps=[
                    ('preprocessor', preprocessor),
                    ('model', XGBRegressor(
                        n_estimators=300,
                        learning_rate=0.1,
                        max_depth=8,
                        min_child_weight=10,
                        subsample=1.0,
                        colsample_bytree=1.0,
                        gamma=0,
                        reg_alpha=0,
                        reg_lambda=1.0,
                        objective='reg:squarederror',
                        random_state=1,
                        n_jobs=-1
                    ))
                ]
              )

In [ ]:
final_xgb_model.fit(X_train, y_train)

In [ ]:
y_test_pred_xgb = final_xgb_model.predict(X_test)

In [ ]:
mae_xgb_test = round(mean_absolute_error(y_test, y_test_pred_xgb), 3)
rmse_xgb_test = round(np.sqrt(mean_squared_error(y_test, y_test_pred_xgb)), 3)
r2_xgb_test = round(r2_score(y_test, y_test_pred_xgb), 3)

print('XGBregressor:')
print('MAE: ', mae_xgb_test)
print('RMSE: ', rmse_xgb_test)
print('R2 score: ', r2_xgb_test)

In [ ]:
if RUN_OLD_MODELS:
  final_hgb_model = Pipeline(
                  steps=[
                      ('preprocessor', hgb_preprocessor),
                      ('model', HistGradientBoostingRegressor(
                          max_iter=700,
                          max_leaf_nodes=63,
                          min_samples_leaf=50,
                          l2_regularization=1.0,
                          random_state=1,
                      ))
                  ]
                )

In [ ]:
if RUN_OLD_MODELS:
  final_hgb_model.fit(X_train, y_train)

In [ ]:
# y_test_pred_hgb = final_hgb_model.predict(X_test)

In [ ]:
# mae_hgb_test = round(mean_absolute_error(y_test, y_test_pred_hgb), 3)
# rmse_hgb_test = round(np.sqrt(mean_squared_error(y_test, y_test_pred_hgb)), 3)
# r2_hgb_test = round(r2_score(y_test, y_test_pred_hgb), 3)

# print('Hist gradient boosting:')
# print('MAE: ', mae_hgb_test)
# print('RMSE: ', rmse_hgb_test)
# print('R2 score: ', r2_hgb_test)

In [ ]:
result_test = pd.DataFrame([
    {
        'Model': 'XGBoost(Test)',
        'MAE(sec)': 199.481,
        'RMSE(sec)': 372.043,
        'R2': 0.746
    },

    {
        'Model': 'HistGradientBoosting(Test)',
        'MAE(sec)': 202.589,
        'RMSE(sec)': 371.212,
        'R2': 0.748
    }
])
result_test

### Residual / Error Analysis

In [ ]:
residual_xgb = y_test - y_test_pred_xgb
# residual_hgb = y_test - y_test_pred_hgb

abs_error_xgb = np.abs(residual_xgb)
# abs_error_hgb = np.abs(residual_hgb)

In [ ]:
residual_summary = pd.DataFrame({
      'Metric': [
          'Mean residual', 'Median residual', 'Std residual', 'MAE', 'RMSE', 'R2'
      ],

      'XGBregressor': [
          residual_xgb.mean(),
          residual_xgb.median(),
          residual_xgb.std(),
          mae_xgb_test,
          rmse_xgb_test,
          r2_xgb_test
      ],

      # 'Hist gradient boosting': [
      #     residual_hgb.mean(),
      #     residual_hgb.median(),
      #     residual_hgb.std(),
      #     mae_hgb_test,
      #     rmse_hgb_test,
      #     r2_hgb_test
      # ]
  })

residual_summary

### Residual Distribution

In [ ]:
plt.figure(figsize=(10, 6))

plt.hist(residual_xgb, bins=100, range=(-1000, 1000), alpha=0.5, label='XGBregressor')

# plt.hist(residual_hgb, bins=100, range=(-1000, 1000), alpha=0.5, label='Hist gradient boosting')

plt.axvline(0, linestyle='--', linewidth=2)
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

plt.scatter(y_test, y_test_pred_xgb, alpha=0.15, s=10)

min_val = np.minimum(y_test.min(), y_test_pred_xgb.min())
max_val = np.maximum(y_test.max(), y_test_pred_xgb.max())

plt.plot([min_val, max_val], [min_val, max_val], linestyle='--', linewidth=2)

### Error By Trip Duration

In [ ]:
error_analysis = pd.DataFrame({
    'actual': y_test.values,
    'xgb_absolute_error': abs_error_xgb,
    # 'hgb_absolute_error': abs_error_hgb,
    'xgb_residual': residual_xgb,
    # 'hgb_residual': residual_hgb
})

In [ ]:
error_analysis['duration_bucket'] = pd.cut(
    error_analysis['actual'],
    bins=[0, 600, 1200, 1800, 3600, 7200, np.inf],
    labels=['<10 min', '10-20 min', '20-30 min', '30-60 min', '1-2 hours', '>2 hours'],
    include_lowest=True
)

In [ ]:
bucket_result = (
     error_analysis.groupby('duration_bucket', observed=True)
     .agg(
         Count=('actual', 'size'),
         XGB_MAE=('xgb_absolute_error', 'mean'),
        # HGB_MAE=('hgb_absolute_error', 'mean'),
         XGB_MEAN_RESIDUAL=('xgb_residual', 'mean'),
        # HGB_MEAN_RESIDUAL=('hgb_residual', 'mean')
     ).reset_index()
 )
bucket_result

In [ ]:
bucket_result['percentage'] = (
     bucket_result['Count'] / len(y_test) * 100
 )
bucket_result

In [ ]:
error_distance = X_test.copy()

error_distance['actual'] = y_test.values
error_distance['xgb_abs_error'] = abs_error_xgb
# error_distance['hgb_abs_error'] = abs_error_hgb
error_distance['xgb_residual'] = residual_xgb
# error_distance['hgb_residual'] = residual_hgb

In [ ]:
error_distance['distance_bucket'] = pd.cut(
    error_distance['distance_km'],
    bins=[0, 1, 2, 5, 10, 20, np.inf],
    labels=[
        '<1 km',
        '1-2 km',
        '2-5 km',
        '5-10 km',
        '10-20 km',
        '>20 km'
    ], include_lowest=True
)

In [ ]:
distance_result = (
    error_distance.groupby('distance_bucket', observed=True)
    .agg(
        Count=('actual', 'size'),
        XGB_MAE=('xgb_abs_error', 'mean'),
        # HGB_MAE=('hgb_abs_error', 'mean'),
        XGB_MEAN_RESIDUAL=('xgb_residual', 'mean'),
        # HGB_MEAN_RESIDUAL=('hgb_residual', 'mean')
    ).reset_index()
)
distance_result

In [ ]:
error_hour = error_distance.copy()

error_hour['pickup_hour'] = X_test['pickup_hour'].values

In [ ]:
hour_result = (
    error_hour.groupby('pickup_hour', observed=True)
    .agg(
        Count=('actual', 'size'),
        XGB_MAE=('xgb_abs_error', 'mean'),
        # HGB_MAE=('hgb_abs_error', 'mean'),
        XGB_MEAN_RESIDUAL=('xgb_residual', 'mean'),
        # HGB_MEAN_RESIDUAL=('hgb_residual', 'mean')
    ).reset_index()
)
hour_result

In [ ]:
error_day = error_distance.copy()

error_day['pickup_dayofweek'] = X_test['pickup_dayofweek'].values

In [ ]:
day_result = (
    error_day.groupby('pickup_dayofweek', observed=True)
    .agg(
        Count=('actual', 'size'),
        XGB_MAE=('xgb_abs_error', 'mean'),
        # HGB_MAE=('hgb_abs_error', 'mean'),
        XGB_MEAN_RESIDUAL=('xgb_residual', 'mean'),
        # HGB_MEAN_RESIDUAL=('hgb_residual', 'mean')
    ).reset_index()
)
day_result

### XGBregressor 3-fold CV

In [ ]:
kf = KFold(n_splits=3, shuffle=True, random_state=1)

In [ ]:
xgb_cv = cross_validate(
    final_xgb_model,
    X_train,
    y_train,
    cv=kf,
    scoring={
        'MAE': 'neg_mean_absolute_error',
        'RMSE': 'neg_root_mean_squared_error',
        'R2': 'r2'
    },
    n_jobs=-1,
    return_train_score=False
)

In [ ]:
xgb_cv_result = {
    'MAE': round(-xgb_cv['test_MAE'].mean(), 3),
    'RMSE': round(-xgb_cv['test_RMSE'].mean(), 3),
    'R2': round(xgb_cv['test_R2'].mean(), 3)
}
xgb_cv_result

### Feature Importance

In [ ]:
feature_names = preprocessor.get_feature_names_out()
print(len(feature_names))
print(feature_names[:20])

In [ ]:
xgb_regressor = final_xgb_model.named_steps['model']

feature_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': xgb_regressor.feature_importances_
})

feature_importance = feature_importance.sort_values('importance', ascending=False).reset_index(drop=True)
feature_importance.head(20)

In [ ]:
feature_importance['original_feature'] = (
    feature_importance['feature']
    .str.replace(r'^(num|cat)__', '', regex=True)
    .str.replace(r'_\d+$', '', regex=True)
)

In [ ]:
aggregated_importance = (
    feature_importance.groupby('original_feature', as_index=False)['importance'].sum()
    .sort_values('importance', ascending=False).reset_index(drop=True)
)
aggregated_importance

In [ ]:
top_15 = aggregated_importance.head(15).sort_values('importance', ascending=True)

In [ ]:
plt.figure(figsize=(10, 6))

plt.barh(top_15['original_feature'], top_15['importance'])

plt.xlabel('Importance')
plt.ylabel('Feature')
plt.title('Top 15 feature importances: XGBregressor')

plt.tight_layout()

plt.savefig('xgb_feature_importance.png', dpi=300, bbox_inches='tight')

plt.show()

### Final Model Selection

In [ ]:
X_train_final = pd.concat([X_train, X_val], axis=0).reset_index(drop=True)
y_train_final = pd.concat([y_train, y_val], axis=0).reset_index(drop=True)

In [ ]:
print(X_train_final.shape, y_train_final.shape, X_test.shape, y_test.shape)

In [ ]:
X_train_final = X_train_final.drop(columns=['pickup_zone', 'dropoff_zone']).copy()
X_test_final = X_test.drop(columns=['pickup_zone', 'dropoff_zone']).copy()

In [ ]:
pickup_kmeans_final = KMeans(n_clusters=50, random_state=1, n_init=10)
dropoff_kmeans_final = KMeans(n_clusters=50, random_state=1, n_init=10)

In [ ]:
pickup_kmeans_final.fit(X_train_final[['pickup_latitude', 'pickup_longitude']])
dropoff_kmeans_final.fit(X_train_final[['dropoff_latitude', 'dropoff_longitude']])

In [ ]:
X_train_final['pickup_zone'] = pickup_kmeans_final.predict(X_train_final[['pickup_latitude', 'pickup_longitude']])
X_train_final['dropoff_zone'] = dropoff_kmeans_final.predict(X_train_final[['dropoff_latitude', 'dropoff_longitude']])

In [ ]:
X_test_final['pickup_zone'] = pickup_kmeans_final.predict(X_test_final[['pickup_latitude', 'pickup_longitude']])
X_test_final['dropoff_zone'] = dropoff_kmeans_final.predict(X_test_final[['dropoff_latitude', 'dropoff_longitude']])

In [ ]:
final_model = Pipeline(
                steps=[
                    ('preprocessor', preprocessor),
                    ('model', XGBRegressor(
                        n_estimators=300,
                        learning_rate=0.1,
                        max_depth=8,
                        min_child_weight=10,
                        subsample=1.0,
                        colsample_bytree=1.0,
                        gamma=0,
                        reg_alpha=0,
                        reg_lambda=1.0,
                        objective='reg:squarederror',
                        random_state=1,
                        n_jobs=-1
                    ))
                ]
              )

In [ ]:
final_model.fit(X_train_final, y_train_final)

In [ ]:
y_pred_final = final_model.predict(X_test_final)

In [ ]:
mae_final = round(mean_absolute_error(y_test, y_pred_final), 3)
rmse_final = round(np.sqrt(mean_squared_error(y_test, y_pred_final)), 3)
r2_final = round(r2_score(y_test, y_pred_final), 3)

print('Final model:')
print('MAE: ', mae_final)
print('RMSE: ', rmse_final)
print('R2 score: ', r2_final)

### Saving The Model

In [ ]:
os.makedirs('models', exist_ok=True)

In [ ]:
joblib.dump(final_model, 'models/final_model.joblib')
joblib.dump(pickup_kmeans_final, 'models/pickup_kmeans.joblib')
joblib.dump(dropoff_kmeans_final, 'models/dropoff_kmeans.joblib')

In [ ]:
print(os.listdir('models'))

### Reusable Prediction Function

In [ ]:
final_model = joblib.load('models/final_model.joblib')
pickup_kmeans_final = joblib.load('models/pickup_kmeans.joblib')
dropoff_kmeans_final = joblib.load('models/dropoff_kmeans.joblib')

In [ ]:
def predict_trip_duration(
    passenger_count,
    pickup_longitude,
    pickup_latitude,
    dropoff_longitude,
    dropoff_latitude,
    pickup_datetime,
    vendor_id,
    store_and_fwd_flag
):
    # --------------------------------------------------
    # 1. Convert datetime
    # --------------------------------------------------
    pickup_datetime = pd.to_datetime(pickup_datetime)

    pickup_hour = pickup_datetime.hour
    pickup_day = pickup_datetime.day
    pickup_dayofweek = pickup_datetime.dayofweek
    pickup_month = pickup_datetime.month

    # --------------------------------------------------
    # 2. Calendar features
    # --------------------------------------------------
    is_weekend = int(pickup_dayofweek >= 5)

    is_rush_hour = int(
        (pickup_dayofweek < 5)
        and (
            7 <= pickup_hour <= 9
            or 16 <= pickup_hour <= 19
        )
    )

    # --------------------------------------------------
    # 3. Cyclic time features
    # --------------------------------------------------
    hour_sin = np.sin(2 * np.pi * pickup_hour / 24)
    hour_cos = np.cos(2 * np.pi * pickup_hour / 24)

    dayofweek_sin = np.sin(
        2 * np.pi * pickup_dayofweek / 7
    )

    dayofweek_cos = np.cos(
        2 * np.pi * pickup_dayofweek / 7
    )

    # --------------------------------------------------
    # 4. Haversine distance
    # --------------------------------------------------
    lat1 = np.radians(pickup_latitude)
    lon1 = np.radians(pickup_longitude)

    lat2 = np.radians(dropoff_latitude)
    lon2 = np.radians(dropoff_longitude)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2)
    c = 2 * np.arcsin(np.sqrt(a))
    distance_km = 6371 * c

    # --------------------------------------------------
    # 5. Manhattan distance
    # --------------------------------------------------
    lat_diff = np.radians(dropoff_latitude - pickup_latitude)
    lon_diff = np.radians(dropoff_longitude - pickup_longitude)

    mean_lat = np.radians((pickup_latitude + dropoff_latitude) / 2)
    lat_distance_km = 6371 * np.abs(lat_diff)
    lon_distance_km = (6371 * np.abs(lon_diff) * np.cos(mean_lat))
    manhattan_distance_km = (lat_distance_km + lon_distance_km)

    # --------------------------------------------------
    # 6. Bearing
    # --------------------------------------------------
    bearing_x = (np.sin(dlon) * np.cos(lat2))
    bearing_y = (np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon))
    bearing = (np.degrees(np.arctan2(bearing_x, bearing_y)) + 360) % 360
    bearing_rad = np.radians(bearing)
    bearing_sin = np.sin(bearing_rad)
    bearing_cos = np.cos(bearing_rad)

    # --------------------------------------------------
    # 7. Route detour ratio
    # --------------------------------------------------
    if distance_km > 0:
        route_detour_ratio = (
            manhattan_distance_km / distance_km
        )
    else:
        route_detour_ratio = 0

    # --------------------------------------------------
    # 8. Predict spatial zones using final KMeans
    # --------------------------------------------------
    pickup_input = pd.DataFrame([{
        'pickup_latitude': pickup_latitude,
        'pickup_longitude': pickup_longitude
    }])

    dropoff_input = pd.DataFrame([{
        'dropoff_latitude': dropoff_latitude,
        'dropoff_longitude': dropoff_longitude
    }])

    pickup_zone = pickup_kmeans_final.predict(pickup_input)[0]
    dropoff_zone = dropoff_kmeans_final.predict(dropoff_input)[0]

    # --------------------------------------------------
    # 9. Create model input
    # --------------------------------------------------
    input_data = pd.DataFrame([{
        'passenger_count': passenger_count,

        'pickup_longitude': pickup_longitude,
        'pickup_latitude': pickup_latitude,

        'dropoff_longitude': dropoff_longitude,
        'dropoff_latitude': dropoff_latitude,

        'distance_km': distance_km,

        'pickup_hour': pickup_hour,
        'pickup_day': pickup_day,
        'pickup_dayofweek': pickup_dayofweek,
        'pickup_month': pickup_month,

        'is_weekend': is_weekend,
        'is_rush_hour': is_rush_hour,

        'manhattan_distance_km': manhattan_distance_km,

        'hour_sin': hour_sin,
        'hour_cos': hour_cos,

        'dayofweek_sin': dayofweek_sin,
        'dayofweek_cos': dayofweek_cos,

        'route_detour_ratio': route_detour_ratio,

        'bearing_sin': bearing_sin,
        'bearing_cos': bearing_cos,

        'vendor_id': vendor_id,
        'store_and_fwd_flag': store_and_fwd_flag,

        'pickup_zone': pickup_zone,
        'dropoff_zone': dropoff_zone
    }])

    # --------------------------------------------------
    # 10. Make prediction
    # --------------------------------------------------
    prediction_seconds = final_model.predict(
        input_data
    )[0]

    # --------------------------------------------------
    # 11. Convert seconds to minutes
    # --------------------------------------------------
    prediction_minutes = prediction_seconds / 60

    return prediction_seconds, prediction_minutes, input_data

In [ ]:
for i in range(1, 5):
    row = test_df.iloc[i]

    function_seconds, function_minutes, _ = predict_trip_duration(
        passenger_count=row["passenger_count"],
        pickup_longitude=row["pickup_longitude"],
        pickup_latitude=row["pickup_latitude"],
        dropoff_longitude=row["dropoff_longitude"],
        dropoff_latitude=row["dropoff_latitude"],
        pickup_datetime=row["pickup_datetime"],
        vendor_id=row["vendor_id"],
        store_and_fwd_flag=row["store_and_fwd_flag"]
    )

    direct_seconds = final_model.predict(X_test_final.iloc[[i]])[0]

    print(f"\nRow {i}")
    print(f"Function prediction: {function_seconds:.6f} seconds")
    print(f"Direct prediction:   {direct_seconds:.6f} seconds")
    print(f"Difference:           {abs(function_seconds - direct_seconds):.10f} seconds")